# Mejores Parámetros

In [1]:
import sys
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "reamember").exists():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

import pandas as pd
import plotly.express as px

EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"

PLOTS_ROOT = EXPERIMENTS_ROOT / "plots"
PLOTS_ROOT.mkdir(exist_ok=True)

In [3]:

def plot_confusion_grid(df, row_parameter=None, fixed_parameters=None, noise=False):
    parameters = ["sigma", "xi", "iota", "kappa"]

    if row_parameter is not None and row_parameter not in parameters:
        raise ValueError(f"row_parameter debe ser uno de {parameters} o None")

    sources = (
        ["registered", "registered_noised"]
        if noise
        else ["registered", "unregistered"]
    )
    cell_labels = [["TP", "FN"], ["TP", "FN"] if noise else ["FP", "TN"]]

    fixed_parameters = fixed_parameters or {}
    memory_domains = sorted(df["memory_domain"].dropna().unique())

    if row_parameter is None:
        row_values = [None]
    else:
        row_values = sorted(df[row_parameter].dropna().unique())

    # Fija los parámetros que no se representan en filas.
    selected_parameters = {}
    for parameter in parameters:
        if parameter == row_parameter:
            continue
        values = sorted(df[parameter].dropna().unique())
        selected_parameters[parameter] = fixed_parameters.get(parameter, values[0])

    titles = [
        f"memory_domain={memory_domain}" if row_index == 0 else ""
        for row_index, _ in enumerate(row_values)
        for memory_domain in memory_domains
    ]

    fig = make_subplots(
        rows=len(row_values),
        cols=len(memory_domains),
        subplot_titles=titles,
        horizontal_spacing=0.025,
        vertical_spacing=0.12,
    )

    for row_index, row_value in enumerate(row_values, start=1):
        for col_index, memory_domain in enumerate(memory_domains, start=1):
            mask = df["memory_domain"].eq(memory_domain)

            for parameter, value in selected_parameters.items():
                mask &= df[parameter].eq(value)

            if row_parameter is not None:
                mask &= df[row_parameter].eq(row_value)

            matches = df.loc[mask]
            if matches.empty:
                continue

            result = matches.iloc[0]
            rates = [
                [
                    result[f"rates_{source}_recognized_rate"],
                    result[f"rates_{source}_unrecognized_rate"],
                ]
                for source in sources
            ]

            fig.add_trace(
                go.Heatmap(
                    z=rates,
                    x=["recognized", "unrecognized"],
                    y=sources,
                    zmin=0,
                    zmax=1,
                    colorscale="Blues",
                    text=cell_labels,
                    texttemplate="%{text}<br>%{z:.3f}",
                    textfont=dict(size=12),
                    showscale=(row_index == 1 and col_index == len(memory_domains)),
                    colorbar_title="Rate",
                ),
                row=row_index,
                col=col_index,
            )

    for row_index, row_value in enumerate(row_values, start=1):
        row_title = (
            f"{row_parameter}={row_value}"
            if row_parameter is not None
            else "Confusion matrix"
        )
        fig.update_yaxes(title_text=row_title, row=row_index, col=1)

    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title="Confusion matrices by memory size",
        width=max(4200, 220 * len(memory_domains)),
        height=330 * len(row_values),
    )
    fig.update_layout(font=dict(size=10))
    fig.update_annotations(font_size=9)
    fig.update_xaxes(tickfont=dict(size=8), title_font=dict(size=9))
    fig.update_yaxes(tickfont=dict(size=8), title_font=dict(size=9))

    return fig


## Words

### Experimento 1

#### Sin Ruido

In [4]:
summary_path = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/word_1024/recognition/recognition_confusion_results_nonoise.json"
)

with open(summary_path, "r") as f:
    we1nn = pd.json_normalize(json.load(f), sep="_")

we1nn["precision"] = we1nn["rates_registered_recognized_rate"] / (we1nn["rates_registered_recognized_rate"] + we1nn["rates_unregistered_recognized_rate"])
we1nn["recall"] = we1nn["rates_registered_recognized_rate"] / (we1nn["rates_registered_recognized_rate"] + we1nn["rates_registered_unrecognized_rate"])

we1nn = we1nn.sort_values(["recall"]).reset_index(drop=True)

In [5]:
we1nn.head()

,dataset,latent_dim,memory_domain,sigma,iota,kappa,xi,registered_source,unregistered_source,test_source,...,counts_unregistered_total,counts_test_total,rates_registered_recognized_rate,rates_registered_unrecognized_rate,rates_unregistered_recognized_rate,rates_unregistered_unrecognized_rate,rates_test_recognized_rate,rates_test_unrecognized_rate,precision,recall
0,npvinHnivqn/EnglishDictionary,1024,8192,0.60,0.1,0.0,0.0,first_half_of_train,second_half_of_train,test,...,50220,11161,0.019056,0.980944,0.001314,0.998686,0.000627,0.999373,0.935484,0.019056
1,npvinHnivqn/EnglishDictionary,1024,8192,0.01,0.1,0.0,0.0,first_half_of_train,second_half_of_train,test,...,50220,11161,0.019056,0.980944,0.001314,0.998686,0.000627,0.999373,0.935484,0.019056
2,npvinHnivqn/EnglishDictionary,1024,8192,0.01,0.1,0.1,0.0,first_half_of_train,second_half_of_train,test,...,50220,11161,0.019056,0.980944,0.001314,0.998686,0.000627,0.999373,0.935484,0.019056
3,npvinHnivqn/EnglishDictionary,1024,8192,0.60,0.1,0.1,0.0,first_half_of_train,second_half_of_train,test,...,50220,11161,0.019056,0.980944,0.001314,0.998686,0.000627,0.999373,0.935484,0.019056
4,npvinHnivqn/EnglishDictionary,1024,8192,0.10,0.1,0.0,0.0,first_half_of_train,second_half_of_train,test,...,50220,11161,0.019056,0.980944,0.001314,0.998686,0.000627,0.999373,0.935484,0.019056


In [6]:
# Plot precision vs parameters

fig = px.scatter(
    we1nn.sort_values("memory_domain").assign(memory_domain=we1nn["memory_domain"].astype(str)), 
    x="memory_domain", 
    y="precision", 
    color="memory_domain",
)
fig.update_layout(
    xaxis_title="Memory Domain",
    yaxis_title="Precision",
    legend_title="Memory Domain"
)
fig.show()
fig.write_image(PLOTS_ROOT / "precision_vs_memory_domain.svg")

In [7]:
def plot_confusion(df, memory_domain, sigma, xi, iota, kappa):
    selected = {
        "memory_domain": memory_domain,
        "sigma": sigma,
        "xi": xi,
        "iota": iota,
        "kappa": kappa,
    }
    mask = pd.Series(True, index=df.index)
    for parameter, value in selected.items():
        mask &= df[parameter] == value

    matches = df.loc[mask]
    if matches.empty:
        print("No hay datos para esta combinación.")
        return

    row = matches.iloc[0]
    rates = [
        [row["rates_registered_recognized_rate"], row["rates_registered_unrecognized_rate"]],
        [row["rates_unregistered_recognized_rate"], row["rates_unregistered_unrecognized_rate"]],
    ]

    fig = go.Figure(go.Heatmap(
        z=rates,
        x=["recognized", "unrecognized"],
        y=["registered", "unregistered"],
        zmin=0,
        zmax=1,
        colorscale="Blues",
        text=[["TP", "FN"], ["FP", "TN"]],
        texttemplate="%{text}<br>%{z:.3f}",
        colorbar_title="Rate",
    ))
    fig.update_layout(
        title="Confusion matrix",
        xaxis_title="Outcome",
        yaxis_title="Source",
    )
    fig.update_yaxes(autorange="reversed")
    fig.show()


parameters = ["memory_domain", "sigma", "xi", "iota", "kappa"]

sliders = {
    parameter: widgets.SelectionSlider(
        options=sorted(we1nn[parameter].dropna().unique()),
        description=parameter,
        continuous_update=False,
    )
    for parameter in parameters
}
widgets.interact(lambda **kwargs: plot_confusion(we1nn, **kwargs), **sliders);

interactive(children=(SelectionSlider(continuous_update=False, description='memory_domain', options=(8, 16, 32…

In [8]:
for param in ['sigma', 'xi', 'iota']:
    figs = plot_confusion_grid(
        we1nn,
        row_parameter=param,
        fixed_parameters={"xi": 0.0, "iota": 0.0, "kappa": 0.0},
        noise=False
    )
    figs.update_layout(title=f"Confusion matrix by {param}")
    figs.show()
    figs.write_image(PLOTS_ROOT / f"word_exp1nn_confusion_grid_by_{param}.svg", format="svg")

### Ruido 0.3%

In [9]:
summary_path = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/word_1024/recognition/recognition_confusion_results_noise_003.json"
)

with open(summary_path, "r") as f:
    we1n3 = pd.json_normalize(json.load(f), sep="_")

we1n3["precision"] = we1n3["rates_registered_recognized_rate"] / (we1n3["rates_registered_recognized_rate"] + we1n3["rates_registered_noised_recognized_rate"])
we1n3["recall"] = we1n3["rates_registered_recognized_rate"] / (we1n3["rates_registered_recognized_rate"] + we1n3["rates_registered_unrecognized_rate"])

we1n3 = we1n3.sort_values(["recall"]).reset_index(drop=True)

we1n3

,dataset,latent_dim,memory_domain,sigma,iota,kappa,xi,registered_source,registered_noised_source,test_source,...,counts_registered_noised_total,counts_test_total,rates_registered_recognized_rate,rates_registered_unrecognized_rate,rates_registered_noised_recognized_rate,rates_registered_noised_unrecognized_rate,rates_test_recognized_rate,rates_test_unrecognized_rate,precision,recall
0,npvinHnivqn/EnglishDictionary,1024,16384,0.10,0.1,0.1,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.024502,0.975498,0.000597,0.999403,0.000896,0.999104,0.976200,0.024502
1,npvinHnivqn/EnglishDictionary,1024,16384,0.10,0.1,0.0,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.024502,0.975498,0.000597,0.999403,0.000896,0.999104,0.976200,0.024502
2,npvinHnivqn/EnglishDictionary,1024,16384,0.60,0.1,0.1,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.024502,0.975498,0.000597,0.999403,0.000896,0.999104,0.976200,0.024502
3,npvinHnivqn/EnglishDictionary,1024,16384,0.60,0.1,0.0,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.024502,0.975498,0.000597,0.999403,0.000896,0.999104,0.976200,0.024502
4,npvinHnivqn/EnglishDictionary,1024,16384,0.01,0.1,0.0,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.024502,0.975498,0.000597,0.999403,0.000896,0.999104,0.976200,0.024502
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,npvinHnivqn/EnglishDictionary,1024,1024,0.60,0.0,0.0,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.986808,0.013192,0.986112,0.013888,0.503320,1.000000
284,npvinHnivqn/EnglishDictionary,1024,1024,0.60,0.0,0.1,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.986808,0.013192,0.986112,0.013888,0.503320,1.000000
285,npvinHnivqn/EnglishDictionary,1024,32,0.10,0.0,0.1,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.999801,0.000199,0.999821,0.000179,0.500050,1.000000
286,npvinHnivqn/EnglishDictionary,1024,64,0.01,0.0,0.0,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.999572,0.000428,0.999373,0.000627,0.500107,1.000000


In [10]:
def plot_confusion_noise(df, memory_domain, sigma, xi, iota, kappa):
    selected = {
        "memory_domain": memory_domain,
        "sigma": sigma,
        "xi": xi,
        "iota": iota,
        "kappa": kappa,
    }
    mask = pd.Series(True, index=df.index)
    for parameter, value in selected.items():
        mask &= df[parameter] == value

    matches = df.loc[mask]
    if matches.empty:
        print("No hay datos para esta combinación.")
        return

    row = matches.iloc[0]
    rates = [
        [row["rates_registered_recognized_rate"], row["rates_registered_unrecognized_rate"]],
        [row["rates_registered_noised_recognized_rate"], row["rates_registered_noised_unrecognized_rate"]],
    ]

    fig = go.Figure(go.Heatmap(
        z=rates,
        x=["recognized", "unrecognized"],
        y=["registered", "registered_noised"],
        zmin=0,
        zmax=1,
        colorscale="Blues",
        text=[["TP", "FN"], ["TP", "FN"]],
        texttemplate="%{text}<br>%{z:.3f}",
        colorbar_title="Rate",
    ))
    fig.update_layout(
        title="Recognition rates by input condition",
        xaxis_title="Outcome",
        yaxis_title="Input condition",
    )
    fig.update_yaxes(autorange="reversed")
    fig.show()


parameters = ["memory_domain", "sigma", "xi", "iota", "kappa"]

sliders = {
    parameter: widgets.SelectionSlider(
        options=sorted(we1n3[parameter].dropna().unique()),
        description=parameter,
        continuous_update=False,
    )
    for parameter in parameters
}
widgets.interact(lambda **kwargs: plot_confusion_noise(we1n3, **kwargs), **sliders);

interactive(children=(SelectionSlider(continuous_update=False, description='memory_domain', options=(8, 16, 32…

In [11]:
for param in ['sigma', 'xi', 'iota']:
    figs = plot_confusion_grid(
        we1n3,
        row_parameter=param,
        fixed_parameters={"xi": 0.0, "iota": 0.0, "kappa": 0.0},
        noise=True
    )
    figs.update_layout(title=f"Confusion matrix by {param}")
    figs.show()
    figs.write_image(PLOTS_ROOT / f"word_exp1n3_confusion_grid_by_{param}.svg", format="svg")

### Experimento 2

#### No Noise

In [12]:
data = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/word_1024/best_params/global_memories_results_nonoise.json"
)

with open(data, "r") as f:
    data = json.load(f)
    we2nn = pd.DataFrame(data)

we2nn.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df = we2nn[we2nn["recognized"] >= 0.9] 
filtered_plot_df.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df

,latent,msize,filling_percent,sigma,xi,iota,kappa,recognized,unrecognized,mean_cosine,mean_euclidean,mean_l2,mean_edit_distance
265,1024,16384,1.0,0.01,0.0,0.0,0.1,1.0,0.0,0.969740,0.043279,0.043279,0.706209
148,1024,512,1.0,0.01,10.0,0.0,0.0,1.0,0.0,0.969660,0.043507,0.043507,0.707016
145,1024,512,1.0,0.01,0.0,0.0,0.1,1.0,0.0,0.969617,0.043542,0.043542,0.706120
216,1024,4096,1.0,0.01,0.0,0.0,0.0,1.0,0.0,0.969517,0.043566,0.043566,0.708091
168,1024,1024,1.0,0.01,0.0,0.0,0.0,1.0,0.0,0.969498,0.043519,0.043519,0.712571
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37,1024,16,1.0,0.60,10.0,0.0,0.1,1.0,0.0,0.674681,0.210856,0.210856,8.032972
12,1024,8,1.0,0.60,10.0,0.0,0.0,1.0,0.0,0.661702,0.216859,0.216859,8.090135
8,1024,8,1.0,0.60,0.0,0.0,0.0,1.0,0.0,0.661658,0.216979,0.216979,8.124541
13,1024,8,1.0,0.60,10.0,0.0,0.1,1.0,0.0,0.660721,0.217122,0.217122,8.089508


In [13]:
filtered_plot_df["msize_label"] = filtered_plot_df["msize"].astype(str)

# Ahora grafiquemos los mejores resultados
fig = px.scatter(
    filtered_plot_df[:10],
    x="mean_cosine",
    y="mean_edit_distance",
    size="recognized",
    color="msize_label",
    hover_data={
        "sigma": True,
        "xi": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Best configurations with >=90% recognized",
)
fig.update_layout(
    xaxis_title="mean cosine",
    yaxis_title="mean edit distance",
    legend_title_text="m",
    width=1200,
    height=600,
)
fig.show()

In [14]:
fig.write_image(PLOTS_ROOT / "word_exp2nn_bestconfigs.svg", format="svg")

#### Noise 0.3%

In [15]:
data = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/word_1024/best_params/global_memories_results_noise_0.003.json"
)

with open(data, "r") as f:
    data = json.load(f)
    we2n3 = pd.DataFrame(data)

we2n3.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df2 = we2n3[we2n3["recognized"] >= 0.9] 
filtered_plot_df2.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df2

,latent,msize,filling_percent,sigma,xi,iota,kappa,recognized,unrecognized,mean_cosine,mean_euclidean,mean_l2,mean_edit_distance
196,1024,2048,1.0,0.01,10.0,0.0,0.0,0.957889,0.042111,0.960578,0.050488,0.050488,1.417828
197,1024,2048,1.0,0.01,10.0,0.0,0.1,0.957889,0.042111,0.960550,0.050552,0.050552,1.414087
49,1024,32,1.0,0.01,0.0,0.0,0.1,0.983245,0.016755,0.960335,0.050817,0.050817,1.420084
173,1024,1024,1.0,0.01,10.0,0.0,0.1,0.989786,0.010214,0.960330,0.050692,0.050692,1.423282
172,1024,1024,1.0,0.01,10.0,0.0,0.0,0.989786,0.010214,0.960325,0.050736,0.050736,1.420024
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,1024,16,1.0,0.60,0.0,0.0,0.1,0.998746,0.001254,0.673759,0.210863,0.210863,8.059298
8,1024,8,1.0,0.60,0.0,0.0,0.0,1.000000,0.000000,0.660834,0.217008,0.217008,8.087627
13,1024,8,1.0,0.60,10.0,0.0,0.1,1.000000,0.000000,0.660510,0.217094,0.217094,8.103306
9,1024,8,1.0,0.60,0.0,0.0,0.1,1.000000,0.000000,0.660243,0.217251,0.217251,8.096407


In [16]:
filtered_plot_df2["msize_label"] = filtered_plot_df2["msize"].astype(str)

# Ahora grafiquemos los mejores resultados
fig = px.scatter(
    filtered_plot_df2[:10],
    x="mean_cosine",
    y="mean_edit_distance",
    size="recognized",
    color="msize_label",
    hover_data={
        "sigma": True,
        "xi": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Best configurations with >=90% recognized",
)
fig.update_layout(
    xaxis_title="mean cosine",
    yaxis_title="mean edit distance",
    legend_title_text="m",
    width=1200,
    height=600,
)
fig.show()

In [17]:
fig.write_image(PLOTS_ROOT / "word_exp2n3_bestconfigs.svg", format="svg")

## Definitions

### Experiment 1

#### No noise

In [18]:
summary_path = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/definition_1024/recognition/recognition_confusion_results_nonoise.json"
)

with open(summary_path, "r") as f:
    de1nn = pd.json_normalize(json.load(f), sep="_")

de1nn["precision"] = de1nn["rates_registered_recognized_rate"] / (de1nn["rates_registered_recognized_rate"] + de1nn["rates_unregistered_recognized_rate"])
de1nn["recall"] = de1nn["rates_registered_recognized_rate"] / (de1nn["rates_registered_recognized_rate"] + de1nn["rates_registered_unrecognized_rate"])

de1nn = de1nn.sort_values(["recall"]).reset_index(drop=True)

In [19]:
parameters = ["memory_domain", "sigma", "xi", "iota", "kappa"]

sliders = {
    parameter: widgets.SelectionSlider(
        options=sorted(de1nn[parameter].dropna().unique()),
        description=parameter,
        continuous_update=False,
    )
    for parameter in parameters
}
widgets.interact(lambda **kwargs: plot_confusion(de1nn, **kwargs), **sliders);

interactive(children=(SelectionSlider(continuous_update=False, description='memory_domain', options=(8, 16, 32…

In [20]:
for param in ['sigma', 'xi', 'iota']:
    figs = plot_confusion_grid(
        de1nn,
        row_parameter=param,
        fixed_parameters={"xi": 0.0, "iota": 0.0, "kappa": 0.0},
        noise=False
    )
    figs.update_layout(title=f"Confusion matrix by {param}")
    figs.show()
    figs.write_image(PLOTS_ROOT / f"definition_exp1nn_confusion_grid_by_{param}.svg", format="svg")

### Noise 0.3%

In [21]:
summary_path = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/definition_1024/recognition/recognition_confusion_results_003.json"
)

with open(summary_path, "r") as f:
    de1n3 = pd.json_normalize(json.load(f), sep="_")

de1n3["precision"] = de1n3["rates_registered_recognized_rate"] / (de1n3["rates_registered_recognized_rate"] + de1n3["rates_registered_noised_recognized_rate"])
de1n3["recall"] = de1n3["rates_registered_recognized_rate"] / (de1n3["rates_registered_recognized_rate"] + de1n3["rates_registered_unrecognized_rate"])

de1n3 = de1n3.sort_values(["recall"]).reset_index(drop=True)

de1n3

,dataset,latent_dim,memory_domain,sigma,iota,kappa,xi,registered_source,registered_noised_source,test_source,...,counts_registered_noised_total,counts_test_total,rates_registered_recognized_rate,rates_registered_unrecognized_rate,rates_registered_noised_recognized_rate,rates_registered_noised_unrecognized_rate,rates_test_recognized_rate,rates_test_unrecognized_rate,precision,recall
0,npvinHnivqn/EnglishDictionary,1024,8192,0.10,0.1,0.0,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.182975,0.817025,0.031641,0.968359,0.032793,0.967207,0.852570,0.182975
1,npvinHnivqn/EnglishDictionary,1024,8192,0.60,0.1,0.0,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.182975,0.817025,0.031641,0.968359,0.032793,0.967207,0.852570,0.182975
2,npvinHnivqn/EnglishDictionary,1024,8192,0.60,0.1,0.1,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.182975,0.817025,0.031641,0.968359,0.032793,0.967207,0.852570,0.182975
3,npvinHnivqn/EnglishDictionary,1024,8192,0.10,0.1,0.1,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.182975,0.817025,0.031641,0.968359,0.032793,0.967207,0.852570,0.182975
4,npvinHnivqn/EnglishDictionary,1024,8192,0.01,0.1,0.1,0.0,clean_train,noised_train,noised_test,...,100440,11161,0.182975,0.817025,0.031641,0.968359,0.032793,0.967207,0.852570,0.182975
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,npvinHnivqn/EnglishDictionary,1024,512,0.60,0.0,0.0,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.997401,0.002599,0.996774,0.003226,0.500650,1.000000
282,npvinHnivqn/EnglishDictionary,1024,512,0.60,0.0,0.1,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.997401,0.002599,0.996774,0.003226,0.500650,1.000000
283,npvinHnivqn/EnglishDictionary,1024,512,0.10,0.0,0.0,0.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.791716,0.208284,0.790610,0.209390,0.558124,1.000000
284,npvinHnivqn/EnglishDictionary,1024,1024,0.10,0.0,0.1,10.0,clean_train,noised_train,noised_test,...,100440,11161,1.000000,0.000000,0.985872,0.014128,0.983962,0.016038,0.503557,1.000000


In [22]:
parameters = ["memory_domain", "sigma", "xi", "iota", "kappa"]

sliders = {
    parameter: widgets.SelectionSlider(
        options=sorted(de1n3[parameter].dropna().unique()),
        description=parameter,
        continuous_update=False,
    )
    for parameter in parameters
}
widgets.interact(lambda **kwargs: plot_confusion_noise(de1n3, **kwargs), **sliders);

interactive(children=(SelectionSlider(continuous_update=False, description='memory_domain', options=(8, 16, 32…

In [23]:
for param in ['sigma', 'xi', 'iota']:
    figs = plot_confusion_grid(
        de1n3,
        row_parameter=param,
        fixed_parameters={"xi": 0.0, "iota": 0.0, "kappa": 0.0},
        noise=True
    )
    figs.update_layout(title=f"Confusion matrix by {param}")
    figs.show()
    figs.write_image(PLOTS_ROOT / f"definition_exp1n3_confusion_grid_by_{param}.svg", format="svg")

### Experimento 2

#### No Noise

In [24]:
data = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/definition_1024/best_params/global_memories_results_nonoise.json"
)

with open(data, "r") as f:
    data = json.load(f)
    de2nn = pd.DataFrame(data)

de2nn.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df = de2nn[de2nn["recognized"] >= 0.9] 
filtered_plot_df.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df

,latent,msize,filling_percent,sigma,xi,iota,kappa,recognized,unrecognized,mean_cosine,mean_euclidean,mean_l2,mean_edit_distance
96,1024,1024,1.0,0.01,0.0,0.0,0.0,1.0,0.0,0.968991,0.045793,0.045793,50.129558
53,1024,4096,1.0,0.01,10.0,0.0,0.1,1.0,0.0,0.968909,0.045828,0.045828,50.173730
24,1024,8192,1.0,0.01,0.0,0.0,0.0,1.0,0.0,0.968903,0.045835,0.045835,50.167548
72,1024,2048,1.0,0.01,0.0,0.0,0.0,1.0,0.0,0.968827,0.045953,0.045953,50.167189
52,1024,4096,1.0,0.01,10.0,0.0,0.0,1.0,0.0,0.968817,0.045957,0.045957,50.175522
...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,1024,16,1.0,0.60,0.0,0.0,0.0,1.0,0.0,0.353350,0.238631,0.238631,102.442075
273,1024,8,1.0,0.60,0.0,0.0,0.1,1.0,0.0,0.335985,0.251114,0.251114,103.120778
277,1024,8,1.0,0.60,10.0,0.0,0.1,1.0,0.0,0.335825,0.251150,0.251150,103.298629
272,1024,8,1.0,0.60,0.0,0.0,0.0,1.0,0.0,0.335798,0.250950,0.250950,103.565541


In [25]:
filtered_plot_df["msize_label"] = filtered_plot_df["msize"].astype(str)

# Ahora grafiquemos los mejores resultados
fig = px.scatter(
    filtered_plot_df[:10],
    x="mean_cosine",
    y="mean_edit_distance",
    size="recognized",
    color="msize_label",
    hover_data={
        "sigma": True,
        "xi": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Best configurations with >=90% recognized",
)
fig.update_layout(
    xaxis_title="mean cosine",
    yaxis_title="mean edit distance",
    legend_title_text="m",
    width=1200,
    height=600,
)
fig.show()

In [26]:
fig.write_image(PLOTS_ROOT / "definition_exp2nn_bestconfigs.svg", format="svg")

#### Noise 0.3%

In [27]:
data = (
    EXPERIMENTS_ROOT
    / "npvinHnivqn-EnglishDictionary/definition_1024/best_params/global_memories_results_noise_0.003.json"
)

with open(data, "r") as f:
    data = json.load(f)
    we2n3 = pd.DataFrame(data)

we2n3.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df2 = we2n3[we2n3["recognized"] >= 0.9] 
filtered_plot_df2.sort_values(["mean_cosine"], inplace=True, ascending=False)
filtered_plot_df2

,latent,msize,filling_percent,sigma,xi,iota,kappa,recognized,unrecognized,mean_cosine,mean_euclidean,mean_l2,mean_edit_distance
384,1024,16384,1.0,0.01,50.0,0.0,0.0,0.933071,0.066929,0.951283,0.059101,0.059101,42.925005
264,1024,8192,1.0,0.01,30.0,0.0,0.0,0.944718,0.055282,0.950993,0.059467,0.059467,43.541445
386,1024,16384,1.0,0.01,50.0,0.0,1.0,0.933071,0.066929,0.950961,0.059278,0.059278,42.940945
265,1024,8192,1.0,0.01,30.0,0.0,0.5,0.944718,0.055282,0.950925,0.059538,0.059538,43.545239
266,1024,8192,1.0,0.01,30.0,0.0,1.0,0.944718,0.055282,0.950751,0.059577,0.059577,43.491749
...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,1024,128,1.0,0.60,0.0,0.0,0.1,0.944001,0.055999,0.359274,0.235008,0.235008,91.684226
109,1024,64,1.0,0.60,10.0,0.0,0.1,1.000000,0.000000,0.359048,0.235494,0.235494,102.002419
128,1024,32,1.0,0.60,0.0,0.0,0.0,0.993639,0.006361,0.358843,0.236216,0.236216,100.093327
132,1024,32,1.0,0.60,10.0,0.0,0.0,1.000000,0.000000,0.357757,0.235957,0.235957,102.005376


In [28]:
filtered_plot_df2["msize_label"] = filtered_plot_df2["msize"].astype(str)

# Ahora grafiquemos los mejores resultados
fig = px.scatter(
    filtered_plot_df2[:10],
    x="mean_cosine",
    y="mean_edit_distance",
    size="recognized",
    color="msize_label",
    hover_data={
        "sigma": True,
        "xi": True,
        "iota": True,
        "kappa": True,
        "latent": True,
        "msize": True,
        "filling_percent": True,
        "recognized": True,
        "unrecognized": True,
    },
    title="Best configurations with >=90% recognized",
)
fig.update_layout(
    xaxis_title="mean cosine",
    yaxis_title="mean edit distance",
    legend_title_text="m",
    width=1200,
    height=600,
)
fig.show()

In [29]:
fig.write_image(PLOTS_ROOT / "definition_exp2n3_bestconfigs.svg", format="svg")